## TON IoT Data

In [256]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# import pandas as pd
# import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report, precision_recall_fscore_support
import matplotlib.pyplot as plt
# import torch.nn as nn
# import torch.nn.functional as F
# import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import math, time, os, datetime, psutil, gc
import random
import warnings

%load_ext autoreload
%autoreload 2
from utility import *
from informer import *
from libs import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data Ingestion Step

In [257]:
file='../data/TON_IoT/linux_proc_mem.csv'

In [258]:
df= pd.read_csv(file)
df.head()
sensor_name='Linux'
col_resp='type'

,Unnamed: 0,ts,PID,TRUN,TSLPI,TSLPU,POLI,NICE,PRI,RTPR,...,VSTEXT,VSIZE,RSIZE,VGROW,RGROW,MEM,CMD_mem,label_mem,type_mem,type_final
0,0,1554218915,3257,0,23,0,norm,0,120,0,...,193.0,2.0,519.1,2.0,519.1,0.14,Web-Content,0.0,normal,normal
1,1,1554218920,1442,0,1,0,norm,0,120,0,...,193.0,2.6,402.0,2.6,402.0,0.11,firefox,0.0,normal,normal
2,2,1554218925,3197,0,63,0,norm,0,120,0,...,2219.0,764.2,290.7,764.2,290.7,0.08,Xorg,0.0,normal,normal
3,3,1554218930,2774,0,8,0,norm,0,120,0,...,3063.0,725.9,116.8,725.9,116.8,0.03,update-manager,0.0,normal,normal
4,4,1554218935,2797,0,5,0,norm,0,120,0,...,1350.0,1.4,109.6,1.4,109.6,0.03,nautilus,0.0,normal,normal


In [259]:
df.columns

Index(['Unnamed: 0', 'ts', 'PID', 'TRUN', 'TSLPI', 'TSLPU', 'POLI', 'NICE',
       'PRI', 'RTPR', 'CPUNR', 'Status', 'EXC', 'State', 'CPU', 'CMD', 'label',
       'type', 'PID_mem', 'MINFLT', 'MAJFLT', 'VSTEXT', 'VSIZE', 'RSIZE',
       'VGROW', 'RGROW', 'MEM', 'CMD_mem', 'label_mem', 'type_mem',
       'type_final'],
      dtype='object')

In [260]:
df['datetime'] = df['ts'].copy()
df['datetime'] = pd.to_datetime(df['datetime'], unit='s')
df.drop(columns=['ts', 'CPUNR', 'NICE', 'PRI', 'RTPR'], axis=1, inplace=True)
num_cols=['CPU', 'MEM', 'VSIZE', 'RSIZE', 'VGROW', 'RGROW', 'VSTEXT', 'MINFLT', 'MAJFLT', 'TRUN', 'TSLPI', 'TSLPU']
cat_cols=['State', 'Status', 'POLI', 'EXC']

In [261]:
# Prepare data
# [df[col].replace('-', 'Missing', inplace=True) for col in cat_cols]
# [df[col].fillna('Missing', inplace=True) for col in cat_cols]

# --- Clean response column ---
df[col_resp] = df[col_resp].astype(str).str.strip()

# --- Extract features ---
dt = df["datetime"]

df["month"]   = dt.dt.month
df["day"]     = dt.dt.day
df["weekday"] = dt.dt.weekday
df["hour"]    = dt.dt.hour
df["minute"]  = dt.dt.minute
df["second"]  = dt.dt.second

df = df.sort_values('datetime').reset_index(drop=True)

df.drop(columns=['datetime'], inplace=True)

time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']

In [262]:
df.pivot_table(index=['month', 'day'], columns=col_resp, aggfunc='size', fill_value=0)

type       ddos   dos  injection  mitm  normal  password
month day                                               
4     2       0     0          0     0    6137         0
      3       0     0          0     0   17280         0
      4       0     0          0     0   17280         0
      5       0     0          0     0   17280         0
      6       0     0          0     0   17280         0
      7       0     0          0     0   17280         0
      8       0     0          0     0   17280         0
      9       0     0          0     0   17280         0
      10      0     0          0     0   17280         0
      11      0     0          0     0   17280         0
      12      0     0          0     0   17280         0
      13      0     0          0     0   17280         0
      14      0     0          0     0   17280         0
      15      0     0          0     0   17280         0
      16      0     0          0     0   17280         0
      17      0     0          0     0   17280         0
      18      0     0          0     0   17280         0
      19      0     0          0     0   17280         0
      20      0     0          0     0   17280         0
      21      0     0          0     0   17280         0
      22      0     0          0     0   17280         0
      23      0     0          0     0   17280         0
      24      0   710          0     0   18008         0
      25   5951  7501       7235     0   27521         0
      26   9535  9288       9288     0   32006      5886
      27   4349  4175       4175     0   55787      4175
      28   9997  9611          0     0   36078      9611
      29   6524  6344          0    56   56896      6344
      30      0     0          0     0   72645         0
5     1       0     0          0     0   48837         0
      2       0     0          0     0   34560         0
      3       0     0          0     0   28901         0
      4       0     0          0     0   17280         0
      5       0     0          0     0   17280         0
      6       0     0          0     0   17280         0
      7       0     0          0     0   17280         0
      8       0     0          0     0   17280         0
      9       0     0          0     0   12498         0

In [263]:
# 1. Prepare Features and Target
X = df[num_cols+cat_cols+time_cols]
y = df[col_resp].str.lower()

In [264]:
df['type_final'].value_counts()
df['type'].value_counts()

type_final
normal       860992
complex       83070
ddos          18162
dos           17537
password      11363
injection      8729
mitm             56
Name: count, dtype: int64

type
normal       879154
dos           37629
ddos          36356
password      26016
injection     20698
mitm             56
Name: count, dtype: int64

In [265]:
# 1. Prepare Features and Target
X = df[num_cols+cat_cols].copy()
y = df[col_resp].str.lower()  # Force lowercase to prevent 'Normal' vs 'normal' mismatch

# 2. Force "normal" to 0, others to 1...N
# Get unique classes and move 'normal' to the front
unique_classes = np.unique(y)
other_classes = [c for c in unique_classes if c != 'normal']
class_names = ['normal'] + sorted(other_classes)

# Create the mapping dictionary
label_map = {name: i for i, name in enumerate(class_names)}
print(f"Manual Mapping: {label_map}")

# Apply the mapping to y
y_encoded = y.map(label_map).values

# 3. Encode categorical features in X
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype(str).str.lower() # Consistency for features too
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

print(f"Encoded classes: {class_names}")

Manual Mapping: {'normal': 0, 'ddos': 1, 'dos': 2, 'injection': 3, 'mitm': 4, 'password': 5}
Encoded classes: ['normal', 'ddos', 'dos', 'injection', 'mitm', 'password']


In [266]:
X.columns

Index(['CPU', 'MEM', 'VSIZE', 'RSIZE', 'VGROW', 'RGROW', 'VSTEXT', 'MINFLT',
       'MAJFLT', 'TRUN', 'TSLPI', 'TSLPU', 'State', 'Status', 'POLI', 'EXC'],
      dtype='object')

In [267]:
from sklearn.model_selection import train_test_split

# 1. First split: 80% for Training+Validation, 20% for Final Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded
)

# 2. Second split: From the 80%, take 15% for Validation (leaving 68% total for Train)
# Adjust test_size=0.1875 because 0.1875 * 0.8 = 0.15 of the total dataset
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=20, stratify=y_temp
)

# 3. Standardize features
scaler = StandardScaler()
X_train_scaled,X_val_scaled,X_test_scaled=X_train.copy(),X_val.copy(),X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val_scaled[num_cols] = scaler.transform(X_val[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# Convert to Numpy
X_train_np = X_train_scaled.to_numpy(dtype='float32')
X_val_np = X_val_scaled.to_numpy(dtype='float32')
X_test_np = X_test_scaled.to_numpy(dtype='float32')

# 4. Create Tensors
X_train_tensor = torch.FloatTensor(X_train_np)
y_train_tensor = torch.LongTensor(y_train).flatten()

X_val_tensor = torch.FloatTensor(X_val_np)
y_val_tensor = torch.LongTensor(y_val).flatten()

X_test_tensor = torch.FloatTensor(X_test_np)
y_test_tensor = torch.LongTensor(y_test).flatten()

# 5. Prepare DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print(f"Train size: {len(X_train_tensor)} | Val size: {len(X_val_tensor)} | Test size: {len(X_test_tensor)}")

Train size: 479956 | Val size: 119989 | Test size: 399964


In [268]:
import psutil
import os
import pynvml
import time

# Initialize system process
process = psutil.Process(os.getpid())

# Initialize GPU handle if available
if torch.cuda.is_available():
    try:
        pynvml.nvmlInit()
        gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    except:
        gpu_handle = None

In [269]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "DCNN" 

# ==========================================
# 4. Build Deep Learning Model
# ==========================================
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) 
            # Note: No Softmax here because CrossEntropyLoss includes it
        )

    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
num_classes = len(class_names)
model = DeepClassifier(input_dim, num_classes).to(device)

# Define Optimizer and Loss
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss() # Equivalent to sparse_categorical_crossentropy

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=2, 
    factor=0.5, 
    verbose=True
)

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [270]:
import time

# ... (Model, Optimizer, and Criterion initialization) ...

print(f"Training started on {device}...")
print("-" * 50)

# Computational Measurement Start
start_stats_train = get_system_stats(device, reset=True)
start_time_train = time.time()

for epoch in range(20):
    _=model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        _=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # .item() is key here: it converts a 1-element tensor to a standard Python float
        train_loss += loss.item() 

    # --- VALIDATION PHASE ---
    _=model.eval()
    val_loss, v_preds, v_true, v_probs = 0.0, [], [], []
    
    with torch.no_grad():
        for v_X, v_y in val_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            
            val_loss += criterion(v_out, v_y).item()
            
            v_probs.extend(torch.softmax(v_out, dim=1).cpu().numpy())
            v_preds.extend(torch.max(v_out, 1)[1].cpu().numpy())
            v_true.extend(v_y.cpu().numpy())

    # --- METRICS & LOGGING ---
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    acc = accuracy_score(v_true, v_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(v_true, v_preds, average='weighted', zero_division=0)
    auroc = roc_auc_score(v_true, v_probs, multi_class='ovr', average='weighted')

    # This single print statement handles the entire epoch output
    print(f"Epoch [{epoch+1:02d}/20] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"STATS >> Acc: {acc:.2f}% | F1: {f1:.4f} | AUROC: {auroc:.4f}")
    print("-" * 50)

print("Training Complete!")

## Training Computational Performance End
train_duration = time.time() - start_time_train
end_stats_train = get_system_stats(device)


Training started on cuda...
--------------------------------------------------
Epoch [01/20] | Train Loss: 0.3826 | Val Loss: 0.3688
STATS >> Acc: 87.93% | F1: 0.8255 | AUROC: 0.8564
--------------------------------------------------
Epoch [02/20] | Train Loss: 0.3658 | Val Loss: 0.3571
STATS >> Acc: 87.97% | F1: 0.8247 | AUROC: 0.8664
--------------------------------------------------
Epoch [03/20] | Train Loss: 0.3598 | Val Loss: 0.3541
STATS >> Acc: 87.95% | F1: 0.8235 | AUROC: 0.8698
--------------------------------------------------
Epoch [04/20] | Train Loss: 0.3565 | Val Loss: 0.3496
STATS >> Acc: 87.96% | F1: 0.8249 | AUROC: 0.8729
--------------------------------------------------
Epoch [05/20] | Train Loss: 0.3538 | Val Loss: 0.3474
STATS >> Acc: 87.97% | F1: 0.8251 | AUROC: 0.8757
--------------------------------------------------
Epoch [06/20] | Train Loss: 0.3518 | Val Loss: 0.3446
STATS >> Acc: 87.97% | F1: 0.8249 | AUROC: 0.8777
------------------------------------------

In [271]:
# 1. Generate Training Classification report
df_metrics_train = get_classification_df(v_true, 
                                         v_preds,
                                         v_probs, 
                                         class_names=class_names,
                                         sensor_name=sensor_name, 
                                         model_name=model_name, 
                                         phase="Training")

#1. Generate computational performance report
df_perf_train = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Training',
    duration=train_duration,
    stats=end_stats_train
)

In [272]:
# ==========================================
# 6. Final Evaluation (Testing Phase)
# ==========================================
print("\n" + "="*50)
print("FINAL TEST EVALUATION STARTED")
print("="*50)

# 1. Reset Measurements for Testing Phase
start_stats_test = get_system_stats(device, reset=True)
start_time_test = time.time()

# 2. Run Inference
_=model.eval()
test_loss, t_preds, t_true, t_probs = 0.0, [], [], []

with torch.no_grad():
    for t_X, t_y in test_loader:
        t_X, t_y = t_X.to(device), t_y.to(device)
        t_out = model(t_X)
        
        test_loss += criterion(t_out, t_y).item()
        
        # Collect probabilities and predictions
        t_probs.extend(torch.softmax(t_out, dim=1).cpu().numpy())
        t_preds.extend(torch.max(t_out, 1)[1].cpu().numpy())
        t_true.extend(t_y.cpu().numpy())

# 3. End Measurements
test_duration = time.time() - start_time_test
end_stats_test = get_system_stats(device)


FINAL TEST EVALUATION STARTED


In [273]:
# 1. Generate Testing Classification report
df_metrics_test = get_classification_df(
    y_true=t_true, 
    y_pred=t_preds, 
    y_probs=t_probs,
    class_names=class_names,
    sensor_name=sensor_name, 
    model_name=model_name, 
    phase="Testing"
)

# 2. Generate Testing Computational performance report
df_perf_test = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Testing',
    duration=test_duration,
    stats=end_stats_test
)

In [274]:
# Final Reports
# Classification
df_metrics = pd.concat([df_metrics_train, df_metrics_test], axis=0)
display(df_metrics)

# Classification
df_perf = pd.concat([df_perf_train, df_perf_test], axis=0)
display(df_perf)

#Saving Classification Report
save_report(df=df_metrics, report_type="classification",output_dir="../output/")

save_report(df=df_perf, report_type="performance",output_dir="../output/")

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.882911,0.996512,0.936278,105498.000000,DCNN,Linux,Training
1,ddos,0.557734,0.058675,0.106180,4363.000000,DCNN,Linux,Training
2,dos,0.535714,0.026578,0.050644,4515.000000,DCNN,Linux,Training
3,injection,0.000000,0.000000,0.000000,2484.000000,DCNN,Linux,Training
4,mitm,0.000000,0.000000,0.000000,7.000000,DCNN,Linux,Training
5,password,0.585470,0.043882,0.081645,3122.000000,DCNN,Linux,Training
6,accuracy,0.880439,0.880439,0.880439,0.880439,DCNN,Linux,Training
7,macro avg,0.426972,0.187608,0.195791,119989.000000,DCNN,Linux,Training
8,weighted avg,0.831954,0.880439,0.831095,119989.000000,DCNN,Linux,Training
9,overall_accuracy,0.880439,0.880439,0.880439,119989.000000,DCNN,Linux,Training


,Metric,Value,Unit,model,sensor,phase
0,Duration,134.3103,Seconds,DCNN,Linux,Training
1,Peak CPU Memory,8751.9800,MB,DCNN,Linux,Training
2,Avg CPU Utilization,7.4000,%,DCNN,Linux,Training
3,Peak GPU Memory,18.0600,MB,DCNN,Linux,Training
4,Avg GPU Utilization,0.0000,%,DCNN,Linux,Training
0,Duration,1.8414,Seconds,DCNN,Linux,Testing
1,Peak CPU Memory,8748.5100,MB,DCNN,Linux,Testing
2,Avg CPU Utilization,7.1000,%,DCNN,Linux,Testing
3,Peak GPU Memory,18.0600,MB,DCNN,Linux,Testing
4,Avg GPU Utilization,0.0000,%,DCNN,Linux,Testing


✅ Report successfully saved to: ../output/classification_report.xlsx | Sheet: DCNN_Linux
✅ Report successfully saved to: ../output/computational_performance.xlsx | Sheet: DCNN_Linux


In [275]:
if(len(np.array(df[col_resp].value_counts()))>2):
    # 1. Get unique types and remove 'normal'
    attack_types = [t for t in df[col_resp].unique() if t != 'normal']

    # 2. Create a mapping where 'normal' is always 0
    mapping = {t: i+1 for i, t in enumerate(attack_types)}
    mapping['normal'] = 0

    # 3. Apply the mapping
    df[col_resp] = df[col_resp].map(mapping)

    print(f"Manual Mapping: {mapping}")

Manual Mapping: {'dos': 1, 'injection': 2, 'ddos': 3, 'password': 4, 'mitm': 5, 'normal': 0}


## Data Preprocessing

In [296]:
time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']
num_cols=['CPU', 'MEM', 'VSIZE', 'RSIZE', 'VGROW', 'RGROW', 'VSTEXT', 'MINFLT', 'MAJFLT', 'TRUN', 'TSLPI', 'TSLPU']
cat_cols=['State', 'Status', 'POLI', 'EXC']
all_features = num_cols + cat_cols + time_cols

In [304]:
# 1. Prepare Features and Target
X = df[num_cols+cat_cols+time_cols].copy()


# 3. Encode categorical features in X
for col in cat_cols:
    X[col] = X[col].astype(str).str.lower() # Consistency for features too
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])


In [308]:
df_new = X
df_new[col_resp] = y_encoded
df_new

,CPU,MEM,VSIZE,RSIZE,VGROW,RGROW,VSTEXT,MINFLT,MAJFLT,TRUN,...,Status,POLI,EXC,month,day,weekday,hour,minute,second,type
0,0.13,0.14,2.0,519.1,2.0,519.1,193.0,859502.0,54.0,0,...,3,2,0,4,2,1,15,28,35,0
1,0.09,0.11,2.6,402.0,2.6,402.0,193.0,247354.0,95.0,0,...,3,2,0,4,2,1,15,28,40,0
2,0.06,0.08,764.2,290.7,764.2,290.7,2219.0,57311.0,33.0,0,...,3,2,0,4,2,1,15,28,45,0
3,0.06,0.03,725.9,116.8,725.9,116.8,3063.0,68006.0,17.0,0,...,3,2,0,4,2,1,15,28,50,0
4,0.04,0.03,1.4,109.6,1.4,109.6,1350.0,72300.0,69.0,0,...,3,2,0,4,2,1,15,28,55,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999904,0.00,0.00,315.3,2076.0,0.0,0.0,350.0,0.0,0.0,0,...,0,2,0,5,9,3,17,21,5,0
999905,0.00,0.00,40212.0,1956.0,0.0,0.0,415.0,0.0,0.0,0,...,0,2,0,5,9,3,17,21,10,0
999906,0.00,0.00,159.8,1704.0,0.0,0.0,36.0,0.0,0.0,0,...,0,2,0,5,9,3,17,21,15,0
999907,0.00,0.00,441.1,1264.0,0.0,0.0,18.0,0.0,0.0,0,...,0,2,0,5,9,3,17,21,20,0


### TRAIN TEST SPLIT

In [309]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader

# --- A. Define Features & Create Sequences ---
feature_cols = [col for col in df_new.columns if col not in 
                ['label', 'type']]

In [310]:
## Setting Parameters
# data loading params
seq_len = 360
batch_size = 32

# Model training params
pred_len = 1
d_ff=512
d_model=512
num_epochs = 50

In [311]:
config = {
    'num_idx': [all_features.index(c) for c in num_cols],
    'cat_idx': [all_features.index(c) for c in cat_cols],
    'time_idx': [all_features.index(c) for c in time_cols],
    # Get unique counts for embeddings (+1 for safety/padding)
    'cat_dims': [df_new[c].nunique() + 1 for c in cat_cols],
    'time_dims': {
        'month': 13, 'day': 32, 'weekday': 7, 
        'hour': 24, 'minute': 60, 'second': 60
    }
}

In [313]:
def create_sequences_fixed(df, feature_list, seq_len=64, y_col='type'):
    X, y = [], []
    
    # Get values for all features
    data = df[feature_list].values
    labels = df[y_col].values
    
    # Slide across the whole dataset
    step = seq_len // 2 
    for i in range(0, len(data) - seq_len + 1, step):
        X.append(data[i : i + seq_len])
        # Get label of the last timestamp in the window
        y.append(labels[i + seq_len - 1])
                
    return np.array(X), np.array(y)

In [ ]:
# 1. Split IDs and create DataFrames (3-way split)
ids = df_new.index
# First split: 60% Train+Val, 40% Test
train_val_ids, test_ids = train_test_split(ids, test_size=0.4, random_state=42, shuffle=True)
# Second split: From that 60%, take 20% for Validation (leaving 48% total for Train)
train_ids, val_ids = train_test_split(train_val_ids, test_size=0.2, random_state=42, shuffle=True)

df_train = df_new[df_new.index.isin(train_ids)]
df_val = df_new[df_new.index.isin(val_ids)]
df_test = df_new[df_new.index.isin(test_ids)]

# 2. Create Sequences
X_train_seq, y_train_seq = create_sequences_fixed(df_train, all_features, seq_len=64)
X_val_seq, y_val_seq = create_sequences_fixed(df_val, all_features, seq_len=64)
X_test_seq, y_test_seq = create_sequences_fixed(df_test, all_features, seq_len=64)

# 3. SMOTE (Apply ONLY to Training data)
n_samples, seq_len, n_features = X_train_seq.shape
X_train_flat = X_train_seq.reshape(n_samples, -1)

# Check the minimum number of samples in any class
min_samples = np.min(np.bincount(y_train_seq))

if min_samples > 1:
    # Set k_neighbors to min(5, min_samples - 1)
    k = min(5, min_samples - 1)
    sm = SMOTE(random_state=20, k_neighbors=k)
    X_res_flat, y_res_encoded = sm.fit_resample(X_train_flat, y_train_seq)
else:
    print("One of your classes has only 1 sample. SMOTE requires at least 2.")
    print("Using RandomOverSampler")
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=20)
    X_res_flat, y_res_encoded = ros.fit_resample(X_train_flat, y_train_seq)

# RE-SHAPE TO 3D IMMEDIATELY
X_train_3D = X_res_flat.reshape(-1, seq_len, n_features).astype(np.float32)
X_val_3D = X_val_seq.astype(np.float32) # Val stays original
X_test_3D = X_test_seq.astype(np.float32)

# 4. Targeted Scaling (Fit on Train, Transform all)
scaler = StandardScaler()

num_indices = config['num_idx']

def scale_3d(data, indices, scaler_obj):
    # Only perform math if we actually have indices to scale
    if len(indices) > 0:
        reshaped = data[:, :, indices].reshape(-1, len(indices))
        data[:, :, indices] = scaler_obj.transform(reshaped).reshape(-1, seq_len, len(indices))
    return data

if(len(num_indices)>0):
    num_train_part = X_train_3D[:, :, num_indices].reshape(-1, len(num_indices))
    scaler.fit(num_train_part)

    X_train_3D = scale_3d(X_train_3D, num_indices, scaler)
    X_val_3D = scale_3d(X_val_3D, num_indices, scaler)
    X_test_3D = scale_3d(X_test_3D, num_indices, scaler)

# Helper to scale 3D volumes
# def scale_3d(data, indices, scaler):
#     reshaped = data[:, :, indices].reshape(-1, len(indices))
#     data[:, :, indices] = scaler.transform(reshaped).reshape(-1, seq_len, len(indices))
#     return data



# 5. Create Tensors and DataLoaders
X_train_tensor = torch.FloatTensor(X_train_3D)
y_train_tensor = torch.LongTensor(y_res_encoded)

X_val_tensor = torch.FloatTensor(X_val_3D)
y_val_tensor = torch.LongTensor(y_val_seq)

X_test_tensor = torch.FloatTensor(X_test_3D)
y_test_tensor = torch.LongTensor(y_test_seq)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print(f"Train: {X_train_tensor.shape} | Val: {X_val_tensor.shape} | Test: {X_test_tensor.shape}")

Train size: 479956 | Val size: 119989 | Test size: 399964


In [106]:
feature_metadata = {
    'num_cols': num_cols,
    'cat_cols': categorical_cols,
    'cat_cardinalities': {col: df[col].nunique() for col in categorical_cols}
}

## Running and saving the models

In [107]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "DCNN" 

# ==========================================
# 4. Build Deep Learning Model
# ==========================================
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) 
            # Note: No Softmax here because CrossEntropyLoss includes it
        )

    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
num_classes = len(class_names)
model = DeepClassifier(input_dim, num_classes).to(device)

# Define Optimizer and Loss
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss() # Equivalent to sparse_categorical_crossentropy

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=2, 
    factor=0.5, 
    verbose=True
)

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [108]:
import time

# ... (Model, Optimizer, and Criterion initialization) ...

print(f"Training started on {device}...")
print("-" * 50)

# Computational Measurement Start
start_stats_train = get_system_stats(device, reset=True)
start_time_train = time.time()

for epoch in range(20):
    _=model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        _=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # .item() is key here: it converts a 1-element tensor to a standard Python float
        train_loss += loss.item() 

    # --- VALIDATION PHASE ---
    _=model.eval()
    val_loss, v_preds, v_true, v_probs = 0.0, [], [], []
    
    with torch.no_grad():
        for v_X, v_y in val_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            
            val_loss += criterion(v_out, v_y).item()
            
            v_probs.extend(torch.softmax(v_out, dim=1).cpu().numpy())
            v_preds.extend(torch.max(v_out, 1)[1].cpu().numpy())
            v_true.extend(v_y.cpu().numpy())

    # --- METRICS & LOGGING ---
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    acc = accuracy_score(v_true, v_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(v_true, v_preds, average='weighted', zero_division=0)
    auroc = roc_auc_score(v_true, v_probs, multi_class='ovr', average='weighted')

    # This single print statement handles the entire epoch output
    print(f"Epoch [{epoch+1:02d}/20] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"STATS >> Acc: {acc:.2f}% | F1: {f1:.4f} | AUROC: {auroc:.4f}")
    print("-" * 50)

print("Training Complete!")

## Training Computational Performance End
train_duration = time.time() - start_time_train
end_stats_train = get_system_stats(device)


Training started on cuda...
--------------------------------------------------
Epoch [01/20] | Train Loss: 0.5155 | Val Loss: 0.4913
STATS >> Acc: 87.96% | F1: 0.8244 | AUROC: 0.7094
--------------------------------------------------
Epoch [02/20] | Train Loss: 0.4879 | Val Loss: 0.4687
STATS >> Acc: 87.96% | F1: 0.8243 | AUROC: 0.7553
--------------------------------------------------
Epoch [03/20] | Train Loss: 0.4723 | Val Loss: 0.4568
STATS >> Acc: 87.93% | F1: 0.8247 | AUROC: 0.7753
--------------------------------------------------
Epoch [04/20] | Train Loss: 0.4634 | Val Loss: 0.4518
STATS >> Acc: 87.96% | F1: 0.8250 | AUROC: 0.7838
--------------------------------------------------
Epoch [05/20] | Train Loss: 0.4589 | Val Loss: 0.4444
STATS >> Acc: 87.98% | F1: 0.8275 | AUROC: 0.7898
--------------------------------------------------
Epoch [06/20] | Train Loss: 0.4558 | Val Loss: 0.4434
STATS >> Acc: 87.95% | F1: 0.8254 | AUROC: 0.7881
------------------------------------------

In [109]:
# 1. Generate Training Classification report
df_metrics_train = get_classification_df(v_true, 
                                         v_preds,
                                         v_probs, 
                                         class_names=class_names,
                                         sensor_name=sensor_name, 
                                         model_name=model_name, 
                                         phase="Training")

#1. Generate computational performance report
df_perf_train = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Training',
    duration=train_duration,
    stats=end_stats_train
)

In [110]:
# ==========================================
# 6. Final Evaluation (Testing Phase)
# ==========================================
print("\n" + "="*50)
print("FINAL TEST EVALUATION STARTED")
print("="*50)

# 1. Reset Measurements for Testing Phase
start_stats_test = get_system_stats(device, reset=True)
start_time_test = time.time()

# 2. Run Inference
_=model.eval()
test_loss, t_preds, t_true, t_probs = 0.0, [], [], []

with torch.no_grad():
    for t_X, t_y in test_loader:
        t_X, t_y = t_X.to(device), t_y.to(device)
        t_out = model(t_X)
        
        test_loss += criterion(t_out, t_y).item()
        
        # Collect probabilities and predictions
        t_probs.extend(torch.softmax(t_out, dim=1).cpu().numpy())
        t_preds.extend(torch.max(t_out, 1)[1].cpu().numpy())
        t_true.extend(t_y.cpu().numpy())

# 3. End Measurements
test_duration = time.time() - start_time_test
end_stats_test = get_system_stats(device)


FINAL TEST EVALUATION STARTED


In [111]:
# 1. Generate Testing Classification report
df_metrics_test = get_classification_df(
    y_true=t_true, 
    y_pred=t_preds, 
    y_probs=t_probs,
    class_names=class_names,
    sensor_name=sensor_name, 
    model_name=model_name, 
    phase="Testing"
)

# 2. Generate Testing Computational performance report
df_perf_test = get_performance_df(
    model_name=model_name,
    sensor_name=sensor_name,
    phase='Testing',
    duration=test_duration,
    stats=end_stats_test
)

In [112]:
# Final Reports
# Classification
df_metrics = pd.concat([df_metrics_train, df_metrics_test], axis=0)
display(df_metrics)

# Classification
df_perf = pd.concat([df_perf_train, df_perf_test], axis=0)
display(df_perf)

#Saving Classification Report
save_report(df=df_metrics, report_type="classification",output_dir="../output/")

save_report(df=df_perf, report_type="performance",output_dir="../output/")

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.881602,0.997830,0.936122,105509.000000,DCNN,Linux,Training
1,dos,0.461538,0.053156,0.095333,4515.000000,DCNN,Linux,Training
2,injection,0.000000,0.000000,0.000000,2484.000000,DCNN,Linux,Training
3,ddos,0.666667,0.007793,0.015406,4363.000000,DCNN,Linux,Training
4,password,0.222222,0.000641,0.001278,3122.000000,DCNN,Linux,Training
5,mitm,0.000000,0.000000,0.000000,7.000000,DCNN,Linux,Training
6,accuracy,0.879633,0.879633,0.879633,0.879633,DCNN,Linux,Training
7,macro avg,0.372005,0.176570,0.174690,120000.000000,DCNN,Linux,Training
8,weighted avg,0.822527,0.879633,0.827258,120000.000000,DCNN,Linux,Training
9,overall_accuracy,0.879633,0.879633,0.879633,120000.000000,DCNN,Linux,Training


,Metric,Value,Unit,model,sensor,phase
0,Duration,134.9676,Seconds,DCNN,Linux,Training
1,Peak CPU Memory,3900.0700,MB,DCNN,Linux,Training
2,Avg CPU Utilization,7.4000,%,DCNN,Linux,Training
3,Peak GPU Memory,18.0500,MB,DCNN,Linux,Training
4,Avg GPU Utilization,0.0000,%,DCNN,Linux,Training
0,Duration,1.8751,Seconds,DCNN,Linux,Testing
1,Peak CPU Memory,3899.5400,MB,DCNN,Linux,Testing
2,Avg CPU Utilization,7.8000,%,DCNN,Linux,Testing
3,Peak GPU Memory,18.0500,MB,DCNN,Linux,Testing
4,Avg GPU Utilization,0.0000,%,DCNN,Linux,Testing


✅ Report successfully saved to: ../output/classification_report.xlsx | Sheet: DCNN_Linux
✅ Report successfully saved to: ../output/computational_performance.xlsx | Sheet: DCNN_Linux


# Informer

In [113]:
print(f"Manual Mapping: {mapping}")

Manual Mapping: {'dos': 1, 'injection': 2, 'ddos': 3, 'password': 4, 'mitm': 5, 'normal': 0}


In [114]:
df_new = X
df_new['type'] = y_encoded
df_new

,CPU,MEM,VSIZE,RSIZE,VGROW,RGROW,VSTEXT,MINFLT,MAJFLT,TRUN,...,TSLPU,State,Status,POLI,EXC,month,day,weekday,hour,type
0,0.513977,10.577293,-0.792888,-0.542993,-0.077000,0.551140,-0.296743,40.063318,0.296801,-0.297943,...,-0.01036,3,3,2,0,4,2,1,15,0
1,0.291604,8.218605,-0.792834,-0.548710,-0.076835,0.414812,-0.296743,11.506466,0.525541,-0.297943,...,-0.01036,3,3,2,0,4,2,1,15,0
2,0.124824,5.859917,-0.723238,-0.554143,0.132931,0.285237,1.149444,2.640914,0.179641,-0.297943,...,-0.01036,3,3,2,0,4,2,1,15,0
3,0.124824,1.928769,-0.726738,-0.562632,0.122383,0.082782,1.751904,3.139838,0.090377,-0.297943,...,-0.01036,3,3,2,0,4,2,1,15,0
4,0.013637,1.928769,-0.792943,-0.562984,-0.077165,0.074400,0.529140,3.340154,0.380486,-0.297943,...,-0.01036,3,3,2,0,4,2,1,15,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,-0.208737,-0.429919,-0.793071,-0.568334,-0.077551,-0.053197,-0.434509,-0.032658,-0.004467,-0.297943,...,-0.01036,3,0,2,0,5,9,3,17,0
999996,-0.208737,-0.429919,-0.793071,-0.568334,-0.077551,-0.053197,-0.434509,-0.032658,-0.004467,-0.297943,...,-0.01036,3,0,2,0,5,9,3,17,0
999997,-0.208737,-0.429919,-0.793071,-0.568334,-0.077551,-0.053197,-0.434509,-0.032658,-0.004467,-0.297943,...,-0.01036,3,0,2,0,5,9,3,17,0
999998,-0.208737,-0.429919,-0.793071,-0.568334,-0.077551,-0.053197,-0.434509,-0.032658,-0.004467,-0.297943,...,-0.01036,3,0,2,0,5,9,3,17,0


In [115]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader

# --- A. Define Features & Create Sequences ---
feature_cols = [col for col in df_new.columns if col not in 
                ['label', 'type']]


In [116]:
## Setting Parameters
# data loading params
seq_len = 360
batch_size = 32

# Model training params
pred_len = 1
d_ff=512
d_model=512
num_epochs = 50

In [117]:
time_cols = ['month', 'day', 'weekday', 'hour', 'minute', 'second']
cat_cols = list(categorical_cols)
num_cols = [col for col in feature_cols if col not in cat_cols+time_cols]
all_features = num_cols + cat_cols + time_cols

In [118]:
# Map for the model to "find" indices and define embedding sizes
config = {
    'num_idx': [all_features.index(c) for c in num_cols],
    'cat_idx': [all_features.index(c) for c in cat_cols],
    'time_idx': [all_features.index(c) for c in time_cols],
    # Get unique counts for embeddings (+1 for safety/padding)
    'cat_dims': [df_new[c].nunique() + 1 for c in cat_cols],
    'time_dims': {
        'month': 13, 'day': 32, 'weekday': 7, 
        'hour': 24, 'minute': 60, 'second': 60
    }
}

In [119]:
def create_sequences_fixed(df, feature_list, seq_len=64, y_col='type'):
    X, y = [], []
    
    # Get values for all features
    data = df[feature_list].values
    labels = df[y_col].values
    
    # Slide across the whole dataset
    step = seq_len // 2 
    for i in range(0, len(data) - seq_len + 1, step):
        X.append(data[i : i + seq_len])
        # Get label of the last timestamp in the window
        y.append(labels[i + seq_len - 1])
                
    return np.array(X), np.array(y)

In [120]:
df_new.columns

Index(['CPU', 'MEM', 'VSIZE', 'RSIZE', 'VGROW', 'RGROW', 'VSTEXT', 'MINFLT',
       'MAJFLT', 'TRUN', 'TSLPI', 'TSLPU', 'State', 'Status', 'POLI', 'EXC',
       'month', 'day', 'weekday', 'hour', 'type'],
      dtype='object')

In [335]:
# 1. Split IDs and create DataFrames (3-way split)
ids = df_new.index
# First split: 60% Train+Val, 40% Test
train_val_ids, test_ids, y_train_val, y_test = train_test_split(ids, y_encoded, test_size=0.4, random_state=42, shuffle=True, stratify=y)

# Second split: From that 60%, take 20% for Validation (leaving 48% total for Train)
train_ids, val_ids, y_train, y_val = train_test_split(train_val_ids,y_train_val, test_size=0.2, random_state=42, shuffle=True, stratify=y_train_val)

print("--- Stratified Split Verification ---")
print(f"Total Unique Classes in Global Dataset: {len(np.unique(y))}")
print(f"Total Unique Classes in Train Set:      {len(np.unique(y_train))}")
print(f"Total Unique Classes in Val Set:        {len(np.unique(y_val))}")
print(f"Total Unique Classes in Test Set:       {len(np.unique(y_test))}")

df_train = df_new[df_new.index.isin(train_ids)]
df_val = df_new[df_new.index.isin(val_ids)]
df_test = df_new[df_new.index.isin(test_ids)]

# 2. Create Sequences
X_train_seq, y_train_seq = create_sequences_fixed(df_train, all_features, seq_len=64)
X_val_seq, y_val_seq = create_sequences_fixed(df_val, all_features, seq_len=64)
X_test_seq, y_test_seq = create_sequences_fixed(df_test, all_features, seq_len=64)

# 3. SMOTE (Apply ONLY to Training data)
n_samples, seq_len, n_features = X_train_seq.shape
X_train_flat = X_train_seq.reshape(n_samples, -1)

# Check the minimum number of samples in any class
min_samples = np.min(np.bincount(y_train_seq))

if min_samples > 1:
    # Set k_neighbors to min(5, min_samples - 1)
    k = min(5, min_samples - 1)
    sm = SMOTE(random_state=20, k_neighbors=k)
    X_res_flat, y_res_encoded = sm.fit_resample(X_train_flat, y_train_seq)
else:
    print("One of your classes has only 1 sample. SMOTE requires at least 2.")
    print("Using RandomOverSampler")
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=20)
    X_res_flat, y_res_encoded = ros.fit_resample(X_train_flat, y_train_seq)

# RE-SHAPE TO 3D IMMEDIATELY
X_train_3D = X_res_flat.reshape(-1, seq_len, n_features).astype(np.float32)
X_val_3D = X_val_seq.astype(np.float32) # Val stays original
X_test_3D = X_test_seq.astype(np.float32)

# 4. Targeted Scaling (Fit on Train, Transform all)
scaler = StandardScaler()

num_indices = config['num_idx']

def scale_3d(data, indices, scaler_obj):
    # Only perform math if we actually have indices to scale
    if len(indices) > 0:
        reshaped = data[:, :, indices].reshape(-1, len(indices))
        data[:, :, indices] = scaler_obj.transform(reshaped).reshape(-1, seq_len, len(indices))
    return data

if(len(num_indices)>0):
    num_train_part = X_train_3D[:, :, num_indices].reshape(-1, len(num_indices))
    scaler.fit(num_train_part)

    X_train_3D = scale_3d(X_train_3D, num_indices, scaler)
    X_val_3D = scale_3d(X_val_3D, num_indices, scaler)
    X_test_3D = scale_3d(X_test_3D, num_indices, scaler)


# 5. Create Tensors and DataLoaders
X_train_tensor = torch.FloatTensor(X_train_3D)
y_train_tensor = torch.LongTensor(y_res_encoded)

X_val_tensor = torch.FloatTensor(X_val_3D)
y_val_tensor = torch.LongTensor(y_val_seq)

X_test_tensor = torch.FloatTensor(X_test_3D)
y_test_tensor = torch.LongTensor(y_test_seq)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

print(f"Train: {X_train_tensor.shape} | Val: {X_val_tensor.shape} | Test: {X_test_tensor.shape}")


--- Stratified Split Verification ---
Total Unique Classes in Global Dataset: 6
Total Unique Classes in Train Set:      6
Total Unique Classes in Val Set:        6
Total Unique Classes in Test Set:       6
One of your classes has only 1 sample. SMOTE requires at least 2.
Using RandomOverSampler


/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


StandardScaler()

Train: torch.Size([65910, 64, 22]) | Val: torch.Size([3748, 64, 22]) | Test: torch.Size([12497, 64, 22])


In [366]:
# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Informer"

# 2. Initialize the Universal Informer Model
# Ensure 'config' is defined as we discussed (containing num_idx, cat_idx, etc.)
num_classes = len(np.unique(y_res_encoded)) # Dynamic class count from SMOTE labels
seq_len = 64
d_model=128
nhead=8
num_layers=3
mask=''
threshold=2.5

In [367]:
if(model_name=="Informer"):
    model = UniversalInformer(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,           # Transformer layers
    dropout=0.1).to(device)

if(model_name=="InformerMask"):
    model = UniversalInformerMask(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,          # Transformer layers
    masking_method=mask,
    threshold=threshold,
    dropout=0.1).to(device)

if(model_name=="CNNInformerMask"):
    model = UniversalInformerCNN(
    config=config,          # The dynamic mapping dictionary
    num_classes=num_classes,
    d_model=d_model,            # Embedding dimension
    nhead=nhead,                # Attention heads
    num_layers=num_layers,           # Transformer layers
    masking_method=mask,
    threshold=threshold,
    dropout=0.1).to(device)

In [368]:

print(f"Universal Informer initialized for {device}.")
print(f"Processing: {len(config['num_idx'])} continuous, "
      f"{len(config['cat_idx'])} categorical, and "
      f"{len(config['time_idx'])} temporal features.")

# 3. Define Loss, Optimizer, and Scheduler
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Adaptive learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=2, 
    factor=0.5, 
    verbose=True
)

Universal Informer initialized for cuda.
Processing: 12 continuous, 4 categorical, and 6 temporal features.


/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


### Train

In [369]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(class_names)
model = UniversalInformer(config, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

print(f"Training on {device}...")

# Computational Measurement Start
start_stats_train = get_system_stats(device, reset=True)
start_time_train = time.time()


# Create a list of all possible label indices [0, 1, 2, ..., num_classes-1]
all_labels = list(range(len(class_names)))

for epoch in range(20):
    _=model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        _=torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # --- Validation Phase ---
    _=model.eval()
    val_loss, val_preds, val_true, val_probs = 0, [], [], []
    with torch.no_grad():
        for v_X, v_y in val_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            val_loss += criterion(v_out, v_y).item()
            val_probs.extend(torch.softmax(v_out, dim=1).cpu().numpy())
            val_preds.extend(torch.max(v_out, 1)[1].cpu().numpy())
            val_true.extend(v_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    # --- Metrics Calculation ---
    acc = accuracy_score(val_true, val_preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(val_true, val_preds, average='weighted', zero_division=0)
    
    # Safe AUROC Calculation
    try:
        auroc = roc_auc_score(
            val_true, 
            val_probs, 
            multi_class='ovr', 
            average='weighted', 
            labels=all_labels  # Crucial: forces alignment with class_names
        )
    except ValueError:
        # Fallback if the batch is truly degenerate (e.g., only 1 class present)
        auroc = 0.0

    print(f"Epoch [{epoch+1:02d}/20] | Loss: {avg_val_loss:.4f} | Acc: {acc:.2f}% | F1: {f1:.4f} | AUROC: {auroc:.4f}")

# End Measurements
train_duration = time.time() - start_time_train
end_stats_train = get_system_stats(device)

Training on cuda...
Epoch [01/20] | Loss: 0.5800 | Acc: 83.08% | F1: 0.8612 | AUROC: 0.9613
Epoch [02/20] | Loss: 0.6492 | Acc: 84.87% | F1: 0.8726 | AUROC: 0.9525
Epoch [03/20] | Loss: 0.5353 | Acc: 88.34% | F1: 0.8971 | AUROC: 0.9636
Epoch [04/20] | Loss: 0.3086 | Acc: 93.06% | F1: 0.9363 | AUROC: 0.9915
Epoch [05/20] | Loss: 0.2157 | Acc: 96.40% | F1: 0.9651 | AUROC: 0.9885
Epoch [06/20] | Loss: 0.2022 | Acc: 96.21% | F1: 0.9635 | AUROC: 0.9931
Epoch [07/20] | Loss: 0.2132 | Acc: 96.74% | F1: 0.9681 | AUROC: 0.9916
Epoch [08/20] | Loss: 0.1932 | Acc: 96.99% | F1: 0.9705 | AUROC: 0.9897
Epoch [09/20] | Loss: 0.2104 | Acc: 96.72% | F1: 0.9676 | AUROC: 0.9883
Epoch [10/20] | Loss: 0.2111 | Acc: 96.69% | F1: 0.9673 | AUROC: 0.9908
Epoch [11/20] | Loss: 0.2058 | Acc: 96.45% | F1: 0.9641 | AUROC: 0.9909
Epoch [12/20] | Loss: 0.2231 | Acc: 96.96% | F1: 0.9695 | AUROC: 0.9926
Epoch [13/20] | Loss: 0.2209 | Acc: 97.12% | F1: 0.9713 | AUROC: 0.9914
Epoch [14/20] | Loss: 0.2142 | Acc: 97.01% |

In [370]:
# 1. Generate the report dictionary
df_metrics_train = get_classification_df(val_true, 
                                         val_preds,
                                         val_probs,
                                         class_names=class_names,
                                         sensor_name=sensor_name, 
                                         model_name=model_name+mask, 
                                         phase="Training")
df_metrics_train

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.995138,0.993026,0.994081,3298.000000,Informer,Linux,Training
1,ddos,0.970803,0.970803,0.970803,137.000000,Informer,Linux,Training
2,dos,0.691176,0.666667,0.678700,141.000000,Informer,Linux,Training
3,injection,0.481481,0.513158,0.496815,76.000000,Informer,Linux,Training
4,mitm,0.000000,0.000000,0.000000,1.000000,Informer,Linux,Training
5,password,0.912621,0.989474,0.949495,95.000000,Informer,Linux,Training
6,accuracy,0.969851,0.969851,0.969851,0.969851,Informer,Linux,Training
7,macro avg,0.675203,0.688855,0.681649,3748.000000,Informer,Linux,Training
8,weighted avg,0.970041,0.969851,0.969887,3748.000000,Informer,Linux,Training
9,overall_accuracy,0.969851,0.969851,0.969851,3748.000000,Informer,Linux,Training


In [371]:
df_perf_train = get_performance_df(
    model_name=model_name+mask,
    sensor_name=sensor_name,
    phase='Training',
    duration=train_duration,
    stats=end_stats_train
)

df_perf_train

,Metric,Value,Unit,model,sensor,phase
0,Duration,103.527,Seconds,Informer,Linux,Training
1,Peak CPU Memory,9896.270,MB,Informer,Linux,Training
2,Avg CPU Utilization,7.700,%,Informer,Linux,Training
3,Peak GPU Memory,150.430,MB,Informer,Linux,Training
4,Avg GPU Utilization,0.000,%,Informer,Linux,Training


In [372]:
import numpy as np
import torch
import torch.nn.functional as F
import time
from sklearn.metrics import accuracy_score, roc_auc_score

# ========================================================
# 1. Setup for Final Evaluation
# ========================================================
print("\n" + "="*50)
print("FINAL TEST EVALUATION STARTED")
print("="*50)

# Load best model weights if you saved them during training
# model.load_state_dict(torch.load('best_model.pth')) 

_=model.eval()
test_loss, test_preds, test_true, test_probs = 0, [], [], []

# 2. Start Computational Performance Tracking
start_stats_test = get_system_stats(device, reset=True)
start_time_test = time.time()

# 3. Inference Loop
with torch.no_grad():
    for t_X, t_y in test_loader:
        t_X, t_y = t_X.to(device), t_y.to(device)
        
        # Forward pass
        t_out = model(t_X)
        
        # Calculate loss (optional for test set, but good for comparison)
        test_loss += criterion(t_out, t_y).item()
        
        # Store results
        test_probs.extend(F.softmax(t_out, dim=1).cpu().numpy())
        test_preds.extend(torch.max(t_out, 1)[1].cpu().numpy())
        test_true.extend(t_y.cpu().numpy())

# 4. End Computational Performance Tracking
test_duration = time.time() - start_time_test
end_stats_test = get_system_stats(device)
avg_test_loss = test_loss / len(test_loader)

# 5. Calculate Model Metrics
test_true = np.array(test_true)
test_probs = np.array(test_probs)
test_preds = np.array(test_preds)

accuracy = accuracy_score(test_true, test_preds) * 100

# ========================================================
# 5b. Robust AUROC Calculation (Handles Missing Classes)
# ========================================================
# Identify exactly what classes are present in this test subset
unique_classes = np.unique(test_true)
num_classes_in_true = len(unique_classes)

# Determine the number of class columns output by the model architecture
if test_probs.ndim == 1:
    num_classes_in_probs = 2
else:
    num_classes_in_probs = test_probs.shape[1]

# Compute AUROC safely based on dimension states
if num_classes_in_probs == 2:
    # Binary Setup: Isolate the positive class probabilities column
    probs_to_score = test_probs[:, 1] if test_probs.ndim > 1 else test_probs
    auroc = roc_auc_score(test_true, probs_to_score)
else:
    # Multi-class Setup
    if num_classes_in_true != num_classes_in_probs:
        print(f"\n⚠️ Mismatch Warning: Test subset has {num_classes_in_true} classes, "
              f"but model outputs {num_classes_in_probs} probabilities. Filtering matrix columns...")
        
        # Slices only the probability columns for classes that exist in test_true
        probs_to_score = test_probs[:, unique_classes]
    else:
        probs_to_score = test_probs

    auroc = roc_auc_score(test_true, probs_to_score, multi_class='ovr', average='weighted')

# ========================================================
# 6. Print Non-Truncated Metrics Output
# ========================================================
print(f"Final Test Accuracy: {accuracy:.2f}%")
print(f"Final Test AUROC: {auroc:.4f}")
print(f"Inference Time: {test_duration:.4f}s")


FINAL TEST EVALUATION STARTED
Final Test Accuracy: 97.01%
Final Test AUROC: 0.9888
Inference Time: 0.2706s


In [373]:
# 1. Generate the report dictionary
df_metrics_test = get_classification_df(test_true, test_preds, test_probs,
                                        class_names=class_names, 
                                        sensor_name=sensor_name, 
                                        model_name=model_name+mask, 
                                        phase="Testing")
df_metrics_test

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.995441,0.993267,0.994353,10991.000000,Informer,Linux,Testing
1,ddos,0.974194,0.970021,0.972103,467.000000,Informer,Linux,Testing
2,dos,0.674312,0.657718,0.665912,447.000000,Informer,Linux,Testing
3,injection,0.458955,0.497976,0.477670,247.000000,Informer,Linux,Testing
4,mitm,0.000000,0.000000,0.000000,1.000000,Informer,Linux,Testing
5,password,0.930748,0.976744,0.953191,344.000000,Informer,Linux,Testing
6,accuracy,0.970073,0.970073,0.970073,0.970073,Informer,Linux,Testing
7,macro avg,0.672275,0.682621,0.677205,12497.000000,Informer,Linux,Testing
8,weighted avg,0.970697,0.970073,0.970349,12497.000000,Informer,Linux,Testing
9,overall_accuracy,0.970073,0.970073,0.970073,12497.000000,Informer,Linux,Testing


In [374]:
df_perf_test = get_performance_df(
    model_name=model_name+mask,
    sensor_name=sensor_name,
    phase='Testing',
    duration=test_duration,
    stats=end_stats_test
)

df_perf_test

,Metric,Value,Unit,model,sensor,phase
0,Duration,0.2706,Seconds,Informer,Linux,Testing
1,Peak CPU Memory,9896.2700,MB,Informer,Linux,Testing
2,Avg CPU Utilization,8.0000,%,Informer,Linux,Testing
3,Peak GPU Memory,45.7600,MB,Informer,Linux,Testing
4,Avg GPU Utilization,0.0000,%,Informer,Linux,Testing


In [375]:
# Final Reports
# Classification
df_metrics = pd.concat([df_metrics_train, df_metrics_test], axis=0)
df_metrics

# Classification
df_perf = pd.concat([df_perf_train, df_perf_test], axis=0)
df_perf

#Saving Classification Report
save_report(df=df_metrics, report_type="classification",output_dir="../output/")

save_report(df=df_perf, report_type="performance",output_dir="../output/")

,class_or_metric,precision,recall,f1-score,support,model,sensor,phase
0,normal,0.995138,0.993026,0.994081,3298.000000,Informer,Linux,Training
1,ddos,0.970803,0.970803,0.970803,137.000000,Informer,Linux,Training
2,dos,0.691176,0.666667,0.678700,141.000000,Informer,Linux,Training
3,injection,0.481481,0.513158,0.496815,76.000000,Informer,Linux,Training
4,mitm,0.000000,0.000000,0.000000,1.000000,Informer,Linux,Training
5,password,0.912621,0.989474,0.949495,95.000000,Informer,Linux,Training
6,accuracy,0.969851,0.969851,0.969851,0.969851,Informer,Linux,Training
7,macro avg,0.675203,0.688855,0.681649,3748.000000,Informer,Linux,Training
8,weighted avg,0.970041,0.969851,0.969887,3748.000000,Informer,Linux,Training
9,overall_accuracy,0.969851,0.969851,0.969851,3748.000000,Informer,Linux,Training


,Metric,Value,Unit,model,sensor,phase
0,Duration,103.5270,Seconds,Informer,Linux,Training
1,Peak CPU Memory,9896.2700,MB,Informer,Linux,Training
2,Avg CPU Utilization,7.7000,%,Informer,Linux,Training
3,Peak GPU Memory,150.4300,MB,Informer,Linux,Training
4,Avg GPU Utilization,0.0000,%,Informer,Linux,Training
0,Duration,0.2706,Seconds,Informer,Linux,Testing
1,Peak CPU Memory,9896.2700,MB,Informer,Linux,Testing
2,Avg CPU Utilization,8.0000,%,Informer,Linux,Testing
3,Peak GPU Memory,45.7600,MB,Informer,Linux,Testing
4,Avg GPU Utilization,0.0000,%,Informer,Linux,Testing


✅ Report successfully saved to: ../output/classification_report.xlsx | Sheet: Informer_Linux
✅ Report successfully saved to: ../output/computational_performance.xlsx | Sheet: Informer_Linux
